# Tutorial 1.7: Evaluating Agents with MLflow

![](images/8_Evaluating-Agents.png)

## LLM-as-Judge Evaluation for GenAI Applications

This notebook covers how to systematically evaluate agents and LLM applications using MLflow's evaluation framework — built-in judges, custom scorers, and third-party libraries like DeepEval.

### What You'll Learn

- Why agent evaluation matters
- MLflow built-in scorers (RelevanceToQuery, Correctness, Guidelines, Safety)
- Creating deterministic custom scorers with the `@scorer` decorator
- Building LLM-as-a-Judge evaluators with `make_judge`
- DeepEval integration for conversational evaluation
- Session-level multi-turn evaluation

### Prerequisites
- Completed notebooks 1.1-1.6
- Understanding of MLflow tracing

### Estimated Time: 30-35 minutes

---
## Step 1: Why Evaluate Agents?

Traces tell you what the agent did; evaluation tells you how well it did it — whether, given a query or task, it behaved as expected. Rigorous, repeated evals are table stakes for any agent, both in development and in production, and they address a range of challenges.

Agent evaluation is critical for:

| Challenge | Solution |
|-----------|----------|
| Non-deterministic outputs | Evaluation over multiple runs |
| Complex reasoning chains | Step-by-step quality assessment |
| Multi-turn conversations | Session-level coherence metrics |
| Safety and compliance | Automated guardrail checking |

### LLM-as-Judge Pattern

Instead of brittle string matching, we use an LLM to evaluate LLM outputs:

```
Agent Output → Judge LLM → Score + Reasoning
```

![LLM as a judge concept](images/llm-as-judge.png)

Out of the box, MLflow provides:
1. **Built-in scorers** — pre-configured judges for common metrics
2. **Custom scorers** — define your own evaluation logic
3. **Third-party integrations** — DeepEval, RAGAS, Phoenix, TruLens, and more

---
## Step 2: Environment Setup

In [1]:
import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names

# Load environment
load_dotenv()

# Configure MLflow
EXPERIMENT_NAME = "07-agent-evaluation"
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

# Check if we are using a Databricks AI Gateway client
use_databricks_provider = is_databricks_ai_gateway_client()
if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    models = get_ai_gateway_model_names()
    JUDGE_MODEL = models[2]
    AGENT_MODEL = models[0]
else:
    # Initialize as an OpenAI client
    client = get_openai_client()
    JUDGE_MODEL = "gpt-5.2"
    AGENT_MODEL = "gpt-5.2"

# Enable autologging
mlflow.openai.autolog()

print("✅ Environment configured for agent evaluation")
print(f"use_databricks_provider: {use_databricks_provider}")
print(f"MLflow tracking: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Agent model : {AGENT_MODEL}")
print(f"Judge model : {JUDGE_MODEL}")

2026/05/02 15:52:39 INFO mlflow.tracking.fluent: Experiment with name '07-agent-evaluation' does not exist. Creating a new experiment.


✅ Environment configured for agent evaluation
use_databricks_provider: False
MLflow tracking: http://localhost:5000
Experiment: 07-agent-evaluation
Agent model : gpt-5.2
Judge model : gpt-5.2


---
## Step 3: Create a Simple Single-Turn Agent for Evaluation

Let's create a simple Q&A agent to evaluate. This is a stepping stone to the next steps, where we use a number of built-in judges followed by custom scorers.

**Note:** I use the terms *scorer* and *judge* interchangeably — they mean the same thing here.

In [2]:
from typing import Any

class SimpleQAAgent:
    """
    A simple Q&A agent for demonstration.
    """
    
    def __init__(self, client:Any, model: str = "gpt-5-mini"):
        self.model = model
        self.client = client
    
    # create a manual trace for the agent with the span type AGENT
    @mlflow.trace(name="qa_agent", span_type="AGENT")
    def answer(self, question: str) -> str:
        """Answer a question."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": """You are a helpful Agent Assistant. Provide concise, accurate answers, 
                no hallucinations, with a focus on MLflow and GenAI."""},
                {"role": "user", "content": question}
            ],
        )
        return response.choices[0].message.content

# Initialize agent 
agent = SimpleQAAgent(client=client, model=AGENT_MODEL)

# Test the agent with a single question
test_question = "What is MLflow's Gen AI and Agent platform capabilities?"
test_response = agent.answer(test_question)
print(f"Question: {test_question}")
print(f"\nResponse: {test_response}")

Question: What is MLflow's Gen AI and Agent platform capabilities?

Response: MLflow’s GenAI and Agent platform capabilities are the set of features that let you **build, evaluate, track, govern, and deploy** LLM- and agent-based applications with the same lifecycle tooling MLflow provides for traditional ML.

## 1) GenAI development & tracking
- **Experiment tracking for LLM apps**: log prompts, model/provider (e.g., OpenAI, Anthropic, local models), parameters (temperature, max tokens), outputs, tool calls, and metadata.
- **Artifacts for reproducibility**: store prompt templates, retrieval configs, system instructions, and evaluation datasets alongside runs.
- **Easy comparison**: run-level organization to compare prompt versions, model variants, and chain/agent changes.

## 2) Evaluation (LLM-native + classical metrics)
- **Automated evaluation workflows** for LLM outputs, often including:
  - **Quality** (helpfulness, correctness, coherence)
  - **Safety** (toxicity, policy violat

Trace(trace_id=tr-b1c1a7583fc08771232e3c6b1c0e9803)

### Quick Evaluation: Is This Response Good?

Now that we have a response, how do we know if it's good? We could read it manually, but that doesn't scale. Instead, let's use MLflow's built-in `RelevanceToQuery` scorer to have an LLM judge evaluate the response.

Since the agent call in Step 3 was traced, we already have a trace containing the inputs and outputs. We can pass it directly to `mlflow.genai.evaluate()` — no need to manually package the data.

In [3]:
import os
from mlflow.genai.scorers import RelevanceToQuery

# Configure judge model URI based on provider
if use_databricks_provider:
    databricks_token = os.environ.get("DATABRICKS_TOKEN")
    ai_gateway_base_url = os.environ.get("AI_GATEWAY_BASE_URL")
    os.environ["OPENAI_API_KEY"] = databricks_token
    os.environ["OPENAI_API_BASE"] = ai_gateway_base_url
    judge_model_uri = f"openai:/{JUDGE_MODEL}"
else:
    judge_model_uri = f"openai:/{JUDGE_MODEL}"

# Grab the trace from our agent call above — it already has inputs and outputs
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())

# Evaluate with built-in RelevanceToQuery scorer
quick_scorer = RelevanceToQuery(model=judge_model_uri)

print("🔄 Evaluating the test response with RelevanceToQuery scorer...\n")

quick_result = mlflow.genai.evaluate(
    data=[trace],
    scorers=[quick_scorer]
)

# Show the result
print("📊 Quick Evaluation Result:")
print("-" * 40)
score = quick_result.metrics.get("relevance_to_query/mean", "N/A")
print(f"   Relevance Score: {score}")
print("\n✅ The LLM judge evaluated our agent's response!")
print("   Now let's learn about all the built-in scorers available...")

🔄 Evaluating the test response with RelevanceToQuery scorer...



2026/05/02 15:52:53 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

📊 Quick Evaluation Result:
----------------------------------------
   Relevance Score: 1.0

✅ The LLM judge evaluated our agent's response!
   Now let's learn about all the built-in scorers available...


---
## Step 4: MLflow Built-in Scorers

MLflow provides 60+ [pre-configured scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/predefined/) for common evaluation needs:

| Scorer | Purpose | Inputs |
|--------|---------|--------|
| `RelevanceToQuery` | Is the response relevant to the question? | inputs, outputs |
| `Correctness` | Is the response factually correct? | outputs, expectations |
| `Guidelines` | Does the response follow specific guidelines? | outputs |
| `Safety` | Is the response safe and appropriate? | outputs |

Let's construct a set of built-in scorers that we'll use downstream to evaluate the agent's responses against an evaluation dataset.

In [4]:
import os
from mlflow.genai.scorers import (
    RelevanceToQuery,
    Correctness,
    Guidelines,
    Safety
)

if use_databricks_provider:
    databricks_token = os.environ.get("DATABRICKS_TOKEN")
    ai_gateway_base_url = os.environ.get("AI_GATEWAY_BASE_URL")
    
    # Try configuring as OpenAI-compatible endpoint
    os.environ["OPENAI_API_KEY"] = databricks_token
    os.environ["OPENAI_API_BASE"] = ai_gateway_base_url
    
    judge_model_uri = f"databricks:/{JUDGE_MODEL}"
    print("🔧 Configured for Databricks AI Gateway")
    print(f"   Base URL: {ai_gateway_base_url}")
    print(f"   Model: {JUDGE_MODEL}")
else:
    judge_model_uri = f"openai:/{JUDGE_MODEL}"

print(f"🔧 Judge model URI: {judge_model_uri}")

# Initialize built-in scorers
relevance_scorer = RelevanceToQuery(model=judge_model_uri)
correctness_scorer = Correctness(model=judge_model_uri)
safety_scorer = Safety(model=judge_model_uri)

#  Let's create a custom guidelines scorer, which will be used downstream to evaluate traces
#  This guidelines judge, like a policy, dictates how the response should be formatted, and what the response 
# should contain or not contain.
guidelines_scorer = Guidelines(
    model=judge_model_uri,
    guidelines=[ """
Response should be appropriately detailed:
- Simple factual questions: < 200 words  
- Technical how-to questions: < 500 words
- Complex architectural questions: < 1000 words
"""
    ],
    name="custom_guidelines"
)

print("\n✅ Built-in scorers initialized:")
print("   - RelevanceToQuery, Correctness, Safety, Guidelines")

🔧 Judge model URI: openai:/gpt-5.2

✅ Built-in scorers initialized:
   - RelevanceToQuery, Correctness, Safety, Guidelines


---
## Step 5: Create Evaluation Dataset

An evaluation dataset is the foundation of systematic agent testing. Without one, you're eyeballing outputs — which doesn't scale and misses edge cases. A dataset gives you repeatable, automated evaluation, so you can catch regressions when you change prompts, models, or agent logic. It also lets you benchmark performance, A/B test configurations, cover known edge cases, and document the behavior you expect.

There are three steps:
 1. Create an evaluation set of questions (and expected answers) to judge against
 2. Create a prediction function that takes a question and returns the LLM's response
 3. Run `mlflow.genai.evaluate()` with the scorers

### How many evaluation pairs?

A rough guide:

- **Minimum viable**: 10-20 pairs to catch obvious issues
- **Development**: 20-50 pairs covering the main use cases
- **Pre-production**: 50-100 pairs including edge cases
- **Production monitoring**: 100-500+ pairs for statistical significance

The dataset below uses 8 examples for tutorial brevity; real evaluations need more coverage.

### Dataset Structure

- `inputs`: the questions/prompts sent to the agent (a dict with your input field names)
- `expectations` (optional): expected answers for correctness checking

**Important:** use `expected_response` as the field name in `expectations` — that's what MLflow's `Correctness` scorer looks for. Alternatively, use `expected_facts` for fact-based evaluation.

In [5]:
# Evaluation dataset focused on MLflow for GenAI and Agent Observability
# NOTE: Use 'expected_response' - this is the field name that MLflow's Correctness scorer expects
from mlflow.genai import create_dataset
eval_dataset = [
    {
        "inputs": {"question": "What is MLflow Tracing and why is it important for GenAI applications?"},
        "expectations": {"expected_response": "MLflow Tracing provides observability for GenAI applications by capturing the complete execution flow including LLM calls, retrieval steps, tool usage, and agent reasoning. It's important because it enables debugging, performance analysis, and understanding of complex AI pipelines."}
    },
    {
        "inputs": {"question": "How does MLflow help with prompt management in GenAI development?"},
        "expectations": {"expected_response": "MLflow's Prompt Registry allows you to version control prompts, tag and search prompt versions, link prompts to experiments, and collaborate with teams. This ensures reproducibility and systematic prompt engineering."}
    },
    {
        "inputs": {"question": "What are spans in MLflow Tracing and what types are available?"},
        "expectations": {"expected_response": "Spans are units of work captured during tracing. MLflow supports span types including LLM (for model calls), RETRIEVER (for RAG retrieval), TOOL (for function calls), AGENT (for agent orchestration), CHAIN (for sequential operations), and EMBEDDING (for vector operations)."}
    },
    {
        "inputs": {"question": "How can you evaluate LLM outputs using MLflow?"},
        "expectations": {"expected_response": "MLflow provides an evaluation framework with built-in scorers like RelevanceToQuery, Correctness, Guidelines, and Safety. You can also create custom scorers using the @scorer decorator or integrate third-party libraries like DeepEval and RAGAS."}
    },
    {
        "inputs": {"question": "What is the LLM-as-Judge pattern and how does MLflow support it?"},
        "expectations": {"expected_response": "LLM-as-Judge uses an LLM to evaluate outputs from another LLM, replacing brittle string matching with intelligent assessment. MLflow supports this through built-in scorers that use configurable judge models to provide scores and reasoning explanations."}
    },
    {
        "inputs": {"question": "How do you track costs and token usage in MLflow for GenAI?"},
        "expectations": {"expected_response": "MLflow automatically logs token usage (prompt tokens, completion tokens, total tokens) and can calculate costs based on model pricing. This data is captured in traces and experiment runs, enabling cost analysis and optimization across different models and configurations."}
    },
    {
        "inputs": {"question": "What frameworks does MLflow integrate with for GenAI auto-tracing?"},
        "expectations": {"expected_response": "MLflow provides auto-tracing for 40+ frameworks including OpenAI, Anthropic, LangChain, LlamaIndex, AWS Bedrock, Google Vertex AI, Cohere, Ollama, DSPy, AutoGen, and CrewAI. Auto-tracing automatically captures LLM calls without manual instrumentation."}
    },
    {
        "inputs": {"question": "How do you implement session-level tracing for multi-turn conversations?"},
        "expectations": {"expected_response": "Key concepts: (1) stable session identifier, (2) tag traces with session_id, (3) filter traces by session, (4) MLflow search capabilities"}
    },
]

# Create and register the dataset. Registering the dataset allows you to use it in other experiments.
# It's stored in the same experiment as the evaluation runs
dataset  = create_dataset(
    name="regression_test_suite",
    experiment_id= mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id,  
    tags={"type": "regression", "priority": "critical"},
)

dataset.merge_records(eval_dataset)

print(f"✅ Evaluation dataset created with {len(eval_dataset)} examples")
print("\n📋 Questions cover:")
print("   - MLflow Tracing fundamentals")
print("   - Prompt management")
print("   - Span types and structure")
print("   - LLM evaluation methods")
print("   - LLM-as-Judge pattern")
print("   - Cost and token tracking")
print("   - Framework integrations")
print("   - Session-level observability")

✅ Evaluation dataset created with 8 examples

📋 Questions cover:
   - MLflow Tracing fundamentals
   - Prompt management
   - Span types and structure
   - LLM evaluation methods
   - LLM-as-Judge pattern
   - Cost and token tracking
   - Framework integrations
   - Session-level observability


### How does predict_fn work?

#### When to use predict_fn

Use `predict_fn` when:

 * Your dataset contains only inputs (and optionally expectations)
 * You need MLflow to call your agent/model to generate the outputs for evaluation

You don't need `predict_fn` when:

 * Your data already contains pre-computed outputs
 * You're passing MLflow traces (which already contain the inputs/outputs)

When `mlflow.genai.evaluate()` runs, it:

1. Iterates through each item in `eval_dataset`
2. Extracts `item["inputs"]` and passes it to `predict_fn(inputs)`
3. The predict function invokes the LLM (or runs some code) to generate the output
4. Your function returns outputs (e.g., `{"response": "..."}`)

Scorers then evaluate using inputs, outputs, and expectations:
 `[inputs, outputs (from the LLM), expectations] --> scorers`

 - **eval_dataset**: defines *what* to test (questions + expected answers)
 - **predict_fn**: defines *how* to get answers (calls your agent)
 - **scorers**: define *how* to judge quality

In [6]:
def predict_fn(question: str) -> dict:
    """
    Prediction function wrapper for evaluation.
    
    Note: mlflow.genai.evaluate() unpacks the 'inputs' dict as keyword arguments,
    so the function signature must match the keys in your dataset's 'inputs' field.
    
    Dataset: {"inputs": {"question": "..."}} 
    Called as: predict_fn(question="...")
    
    Args:
        question: The question string (unpacked from inputs dict)
        
    Returns:
        Dictionary with 'response' key
    """
    response = agent.answer(question)
    return {"response": response}

print("✅ Prediction function defined")
print("   Signature: predict_fn(question: str) -> dict")

✅ Prediction function defined
   Signature: predict_fn(question: str) -> dict


---
## Step 6: Run Evaluation with Built-in Scorers

In [7]:
# Run evaluation
print("🔄 Running evaluation with built-in scorers...\n")

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        relevance_scorer,
        correctness_scorer,
        safety_scorer,
        guidelines_scorer
    ]
)

print("\n✅ Evaluation complete!")

# Display metrics
print("\n📊 Metrics Summary:")
print("-" * 50)
if results.metrics:
    for metric_name, value in results.metrics.items():
        if isinstance(value, float):
            print(f"  {metric_name}: {value:.3f}")
        else:
            print(f"  {metric_name}: {value}")
else:
    print("  No metrics returned")
    print("\n⚠️  Scorers returned None - this usually means the judge model call failed.")
    print("  Check the 'error_message' or similar columns above for details.")

2026/05/02 15:52:57 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


🔄 Running evaluation with built-in scorers...



Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]


✅ Evaluation complete!

📊 Metrics Summary:
--------------------------------------------------
  safety/mean: 1.000
  relevance_to_query/mean: 1.000
  custom_guidelines/mean: 0.375
  correctness/mean: 0.250


---
## Step 7: Custom Scorers -- Deterministic and LLM-as-a-Judge

Beyond the built-in scorers, MLflow offers two approaches for building custom evaluation logic:

| Dimension | Deterministic (`@scorer`) | LLM-as-a-Judge (`make_judge`) |
|-----------|--------------------------|-------------------------------|
| **Evaluation logic** | Hand-coded Python rules (string checks, length validation, regex) | An LLM reads the trace and applies natural-language judgment |
| **Cost** | Free -- no LLM calls | One LLM inference per evaluation |
| **Best for** | Format validation, keyword presence, structural checks | Nuanced quality assessment, reasoning evaluation, subjective criteria |
| **Data access** | `outputs`, `expectations` dicts | Full execution trace via `{{ trace }}` template variable |
| **Latency** | Milliseconds | Seconds (depends on judge model) |
| **Determinism** | Fully deterministic -- same input always produces same result | Non-deterministic -- LLM may vary across runs |

In practice, **combine both**: use deterministic scorers for cheap guardrails and structural checks, and LLM judges for the assessments that genuinely need reasoning. Let's build both types.

### Step 7(a): Deterministic Scorers with `@scorer` Decorator

The `@scorer` decorator wraps any Python function as an evaluation scorer. These are **deterministic** -- they execute pure Python logic with no LLM calls, making them free, fast, and fully reproducible.

Use deterministic scorers for:
- **Format validation** (response length, structure)
- **Keyword presence** (checking expected terms appear in the response)
- **Guardrail checks** (detecting hallucination markers, prohibited content)

In [8]:
from mlflow.genai import scorer

@scorer
def response_length_check(outputs: dict) -> bool:
    """
    Check if response is within acceptable length.
    Returns True if response is between 200 and 500 characters.
    """
    response = outputs.get("response", "")
    length = len(response)
    return 20 <= length <= 500

@scorer
def contains_keywords(outputs: dict, expectations: dict) -> bool:
    """
    Check if response contains key terms from expected answer.
    """
    response = outputs.get("response", "").lower()
    # Use 'expected_response' to match the dataset field name
    expected = expectations.get("expected_response", "").lower()
    
    # Extract key words (simple approach)
    key_words = [word for word in expected.split() if len(word) > 4]
    
    # Check if at least 30% of key words are present
    # If no keywords to check, fail conservatively (may indicate data issue)
    if not key_words:
        return False
    
    matches = sum(1 for word in key_words if word in response)
    return matches / len(key_words) >= 0.3

@scorer
def no_hallucination_markers(outputs: dict) -> bool:
    """
    Check for common hallucination markers.
    """
    response = outputs.get("response", "").lower()
    
    hallucination_markers = [
        "i think",
        "i believe",
        "probably",
        "might be",
        "i'm not sure",
        "as far as i know"
    ]
    
    return not any(marker in response for marker in hallucination_markers)

print("✅ Custom scorers defined:")
print("   - response_length_check")
print("   - contains_keywords")
print("   - no_hallucination_markers")

✅ Custom scorers defined:
   - response_length_check
   - contains_keywords
   - no_hallucination_markers


### Run Evaluation with Deterministic Scorers

In [9]:
# Run evaluation with custom scorers
print("🔄 Running evaluation with custom scorers...\n")

custom_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        response_length_check,
        contains_keywords,
        no_hallucination_markers
    ]
)

print("\n✅ Custom evaluation complete!")
print("\nCustom Metrics Summary:")
print("-" * 40)
for metric_name, value in custom_results.metrics.items():
    if isinstance(value, float):
        print(f"  {metric_name}: {value:.3f}")
    else:
        print(f"  {metric_name}: {value}")

2026/05/02 15:53:28 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


🔄 Running evaluation with custom scorers...



Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]


✅ Custom evaluation complete!

Custom Metrics Summary:
----------------------------------------
  response_length_check/mean: 0.125
  contains_keywords/mean: 0.750
  no_hallucination_markers/mean: 1.000


### Step 7(b): LLM-as-a-Judge with `make_judge`

When evaluation requires **nuanced judgment** rather than deterministic rules, use `make_judge` from `mlflow.genai.judges`. This creates an LLM-powered judge that receives the **full execution trace** -- inputs, outputs, all spans -- and applies your evaluation criteria.

Key concepts:
- **`{{ trace }}`** is a required template variable in instructions. At evaluation time, it gets replaced with the serialized trace so the judge LLM can inspect the entire execution.
- The judge returns a **`Feedback`** object with `.value` (the rating) and `.rationale` (the explanation).
- You configure which LLM serves as the judge via the `model` parameter (LiteLLM format).

Let's create a judge that evaluates the **explanation quality** of our QA agent's responses.

In [10]:
from mlflow.genai.judges import make_judge

# Create an LLM-as-a-Judge evaluator
explanation_quality_judge = make_judge(
    name="explanation_quality",
    instructions=(
        "Analyze the agent's response in {{ trace }}.\n\n"
        "Evaluate the explanation quality on these criteria:\n"
        "1. **Clarity**: Is the response well-structured and easy to understand?\n"
        "2. **Completeness**: Does it cover the key aspects of the question?\n"
        "3. **Technical Accuracy**: Are technical terms used correctly?\n"
        "4. **Conciseness**: Is it appropriately detailed without unnecessary verbosity?\n\n"
        "Rate as 'yes' if the explanation meets all criteria, 'no' otherwise.\n"
        "Provide specific evidence from the response for your rating."
    ),
    model=f"openai:/{JUDGE_MODEL}",
)

print("✅ LLM-as-a-Judge created: 'explanation_quality'")
print("   Judge model: openai:/{JUDGE_MODEL}")
print("   Template variable: {{{{ trace }}}} (auto-populated with full trace)")

✅ LLM-as-a-Judge created: 'explanation_quality'
   Judge model: openai:/{JUDGE_MODEL}
   Template variable: {{{{ trace }}}} (auto-populated with full trace)


In [11]:
# Generate a fresh trace by calling the agent
test_question_for_judge = "What is MLflow Tracing and why is it important for GenAI applications?"
print(f"Question: {test_question_for_judge}\n")

response = agent.answer(test_question_for_judge)
print(f"Agent Response: {response[:500]} ...\n")

# Retrieve the trace and evaluate using mlflow.genai.evaluate() — same pattern as cell 7
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())
print(f"Trace ID: {trace.info.trace_id}")
print("Running LLM-as-a-Judge evaluation...\n")

judge_result = mlflow.genai.evaluate(
    data=[trace],
    scorers=[explanation_quality_judge]
)

print("=" * 50)
print("Judge Evaluation Result:")
print("-" * 50)
for metric_name, value in judge_result.metrics.items():
    if isinstance(value, float):
        print(f"  {metric_name}: {value:.3f}")
    else:
        print(f"  {metric_name}: {value}")
print("=" * 50)

Question: What is MLflow Tracing and why is it important for GenAI applications?

Agent Response: **MLflow Tracing** is MLflow’s capability for capturing and logging **detailed execution traces** of GenAI application runs—especially LLM-powered workflows. A trace typically records the **tree of spans** for a request (e.g., prompt creation → model call → tool/function calls → retrieval → post-processing), along with timing, inputs/outputs (when safe), metadata, and errors.

### What it provides
- **End-to-end observability** for an LLM request: see every step and dependency in the chain.
- ** ...

Trace ID: tr-47794de12fc6c6f577f0016d33ca99f2
Running LLM-as-a-Judge evaluation...



Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

Judge Evaluation Result:
--------------------------------------------------


Trace(trace_id=tr-47794de12fc6c6f577f0016d33ca99f2)

**When to use which approach:**
- Use **`@scorer`** (Step 7a) when you can express your evaluation as deterministic Python code -- it's free, fast, and fully reproducible.
- Use **`make_judge`** (Step 7b) when evaluation requires understanding context, nuance, or subjective quality -- the LLM judge can reason about the full trace.
- In practice, **combine both**: use deterministic scorers for structural checks and LLM judges for quality assessment.

---
## Step 8: DeepEval Integration

MLflow integrates with [DeepEval](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/third-party/deepeval/) for multi-turn conversational evaluation metrics.

| DeepEval Scorer | Purpose |
|-----------------|----------|
| `ConversationCompleteness` | Did the conversation achieve its goal? |
| `KnowledgeRetention` | Does the agent remember context? |
| `TopicAdherence` | Does the agent stay on topic? |
| `Toxicity` | Is the agent's response harmful or toxic in tone? |

### Understanding `threshold`

Each DeepEval scorer returns a continuous score between 0 and 1. The `threshold` parameter sets the **pass/fail cutoff** — scores `>= threshold` pass, scores below it fail. For example, with `threshold=0.7`, a completeness score of 0.65 fails while 0.75 passes. The aggregate metrics (e.g., `conversation_completeness/mean`) reflect the pass rate across all evaluated examples. The underlying raw scores remain available for more granular analysis.

In [12]:
from mlflow.genai.scorers.deepeval import (
    ConversationCompleteness,
    KnowledgeRetention,
    TopicAdherence,
    Toxicity,
)

# Initialize DeepEval scorers
jude_model_uri = f"openai:/{JUDGE_MODEL}"

completeness_scorer = ConversationCompleteness(model=jude_model_uri, threshold=0.7, include_reason=True)
retention_scorer = KnowledgeRetention(model=jude_model_uri, threshold=0.7, include_reason=True)
toxicity_scorer = Toxicity(model=jude_model_uri, threshold=0.7, include_reason=True)
topic_scorer = TopicAdherence(model=jude_model_uri, threshold=0.7, include_reason=True, relevant_topics=["MLflow", "machine learning", "AI", "data science", "genai", "agent", "observability", "prompt engineering", "prompt management", "prompt registry", "experiment tracking"])


print("✅ DeepEval scorers initialized:")
print("   - ConversationCompleteness")
print("   - KnowledgeRetention")
print("   - TopicAdherence")
print("   - Toxicity")
print("\n Next, let's evaluate a multi-turn conversation using these DeepEval scorers")

✅ DeepEval scorers initialized:
   - ConversationCompleteness
   - KnowledgeRetention
   - TopicAdherence
   - Toxicity

 Next, let's evaluate a multi-turn conversation using these DeepEval scorers


---
## Step 9: Multi-Turn Conversation Agent

Session-level evaluation needs an agent that handles conversations. A session is a series of turns, where each turn is a single trace. Throughout the session, the agent keeps the context of each turn in its memory — which is what scorers like `KnowledgeRetention` rely on.

In [13]:
import uuid

class ConversationalAgent:
    """
    An agent that maintains conversation history for multi-turn interactions.
    """
    
    def __init__(self, client: Any, model: str = "gpt-5-mini"):
        self.model = model
        self.client = client
        self.conversation_history = []
        self.session_id = str(uuid.uuid4())
    
    def reset(self):
        """Reset conversation history and start new session."""
        self.conversation_history = []
        self.session_id = str(uuid.uuid4())
    
    @mlflow.trace(name="conversational_agent", span_type="AGENT")
    def chat(self, user_message: str) -> str:
        """
        Send a message and get a response, maintaining history.
        """
        # Tag trace with session ID for grouping
        mlflow.update_current_trace(metadata={
            "mlflow.trace.session": self.session_id,
            "turn_number": len(self.conversation_history) // 2 + 1
        })
        
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Prepare messages with system prompt. Add previous context to the conversation.
        messages = [
            {"role": "system", 
            "content": """You are a helpful MLflow expert assistant. Answer questions about MLflow clearly, accurately,
                        concisely, without hallucinations, and accurately. Remember previous context in the conversation."""}
        ] + self.conversation_history
        
        # Get response from the Agent LLM
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.7,
        )
        
        assistant_message = response.choices[0].message.content
        
        # Add to history
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message

print("✅ ConversationalAgent defined with session tracking")

✅ ConversationalAgent defined with session tracking


### Simulate a multi-turn conversation

In [14]:
# Simulate a multi-turn conversation
conv_agent = ConversationalAgent(client=client, model=AGENT_MODEL)

conversation_turns = [
    "What is MLflow for GenAI?",
    "What are its main GenAI main components?",
    "Tell me more about the Tracing component.",
    "How does it compare to other tools?",
    "What is the difference between Tracing and Tracking?",
    "How do I get started with MLflow for GenAI?",
]

print("🗣️ Multi-Turn Conversation\n")
print("=" * 60)

for i, user_msg in enumerate(conversation_turns, 1):
    print(f"\n[Turn {i}]")
    print(f"User: {user_msg}")
    response = conv_agent.chat(user_msg)
    print(f"Agent: {response}")

print("\n" + "=" * 60)
print(f"\n✅ Conversation complete (Session: {conv_agent.session_id[:8]}...)")

2026/05/02 15:54:15 WARNING mlflow.tracing.fluent: Found non-string values in metadata. Non-string values in metadata will automatically be stringified when the trace is logged. Non-string items: {'turn_number': 1}


🗣️ Multi-Turn Conversation


[Turn 1]
User: What is MLflow for GenAI?


2026/05/02 15:54:21 WARNING mlflow.tracing.fluent: Found non-string values in metadata. Non-string values in metadata will automatically be stringified when the trace is logged. Non-string items: {'turn_number': 2}


Agent: MLflow for GenAI is MLflow’s set of features for building, evaluating, tracking, and deploying **generative AI applications** (LLM apps), not just traditional ML models. It extends MLflow’s experiment tracking and model management to GenAI-specific artifacts like prompts, LLM configurations, and evaluation results.

Key pieces include:

- **Prompt & LLM tracking**  
  Log prompts, system messages, model/provider (e.g., OpenAI, Anthropic, local models), parameters (temperature, max tokens), retrieved context, and outputs—so you can reproduce and compare runs.

- **GenAI evaluation (LLM evaluation)**  
  Evaluate LLM outputs with built-in workflows: compare model/prompt variants, run automated metrics (including LLM-as-a-judge style scoring where appropriate), and track results over datasets.

- **Prompt Engineering / Prompt Registry (where supported)**  
  Manage versions of prompts and their metadata similarly to how you’d manage model versions—making it easier to iterate and ro

2026/05/02 15:54:27 WARNING mlflow.tracing.fluent: Found non-string values in metadata. Non-string values in metadata will automatically be stringified when the trace is logged. Non-string items: {'turn_number': 3}


...

2026/05/02 15:54:53 WARNING mlflow.tracing.fluent: Found non-string values in metadata. Non-string values in metadata will automatically be stringified when the trace is logged. Non-string items: {'turn_number': 5}


Agent: MLflow Tracing overlaps with “LLM observability” tools (LangSmith, Langfuse, Arize Phoenix, Honeycomb/OpenTelemetry-based setups), but it’s positioned differently: it’s meant to integrate **directly into the MLflow lifecycle** (tracking/eval/registry/serving) rather than be a standalone observability product.

### Key comparison dimensions

#### 1) Scope: LLM app lifecycle vs observability-only
- **MLflow Tracing**: Best when you want traces tied to **experiments, evaluations, and deployed MLflow models** in one place.
- **LangSmith / Langfuse / Phoenix**: Often deeper focus on **prompt/agent debugging workflows**, dataset curation, annotation queues, and LLM-specific UX.

#### 2) Integration with MLflow artifacts (runs, evals, registry)
- **MLflow Tracing**: Strong advantage if you already use MLflow Tracking + Model Registry. You can connect:
  - a trace → the exact prompt/model/version/config logged in a run
  - evaluation failures → the traces behind them
  - production inci

2026/05/02 15:55:00 WARNING mlflow.tracing.fluent: Found non-string values in metadata. Non-string values in metadata will automatically be stringified when the trace is logged. Non-string items: {'turn_number': 6}


Agent: **Tracking** and **Tracing** solve different problems in MLflow:

### MLflow Tracking (experiment tracking)
- **Purpose:** Record and compare *experiments* and *versions*.
- **Unit of organization:** a **run** (an experiment run).
- **What you log:** parameters, metrics, artifacts, and metadata—e.g., model name, temperature, prompt template/version, evaluation scores, datasets, files.
- **Typical cadence:** per experiment / per training or evaluation job (minutes–hours), or per batch evaluation.
- **Primary question it answers:** *“Which configuration performed best, and can I reproduce it?”*

### MLflow Tracing (observability)
- **Purpose:** Debug and observe *individual executions* of an app/pipeline.
- **Unit of organization:** a **trace** made of **spans** (nested steps).
- **What you capture:** step-by-step inputs/outputs, timing, errors/retries—e.g., retrieval step, prompt construction, LLM call, tool call, post-processing.
- **Typical cadence:** per request (milliseconds–

[Trace(trace_id=tr-5285b9afae09082de54a11490716ccf9), Trace(trace_id=tr-2470ade6ccda80a263e9c5e235876f61), Trace(trace_id=tr-0379ae013f0c3f9f25cd24056b60cd5e), Trace(trace_id=tr-f5b5cc1f3cda648e97e8a51b67a08c89), Trace(trace_id=tr-3fc420067c92394117d33c3b77869e90), Trace(trace_id=tr-a488ff5ece3e095d3efdcb9e719d8fd0)]

---
## Step 10: Session-Level Evaluation with DeepEval

In [15]:
# Search for traces from this session
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

session_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{conv_agent.session_id}'"
)

print(f"📊 Found {len(session_traces)} traces for session {conv_agent.session_id[:8]}...")

📊 Found 6 traces for session 3fd00bb4...


### Run the evaluation

In [16]:
with mlflow.start_run(run_name="DeepEval") as run:
    deepeval_results = mlflow.genai.evaluate(
        data= session_traces,
        # My DeepEval scorers defined above
        scorers=[
            completeness_scorer,
            retention_scorer,
            topic_scorer,
            toxicity_scorer
        ]
    )

print("\n✅ DeepEval session evaluation complete!")
print("\nDeepEval Metrics:")
print("-" * 40)
for metric_name, value in deepeval_results.metrics.items():
    if isinstance(value, float):
        print(f"  {metric_name}: {value:.3f}")
    else:
        print(f"  {metric_name}: {value}")

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?]


✅ DeepEval session evaluation complete!

DeepEval Metrics:
----------------------------------------
  Toxicity/mean: 1.000
  ConversationCompleteness/mean: 1.000
  KnowledgeRetention/mean: 0.000
  TopicAdherence/mean: 1.000


---
## Step 11: Viewing Evaluation Results in MLflow UI

### Experiments View

Navigate to: http://localhost:5000  
Select the **"07-agent-evaluation"** experiment.

You'll see:
- Session-level and evaluation runs with scorer metrics
- Pass/fail rates for each scorer
- Detailed reasoning from LLM judges

### Evaluation Results

Each run includes:
- Metric values (0-1 scores or boolean)
- Judge reasoning explanations
- Input/output pairs
- Artifacts with detailed results

### Key Metrics to Monitor

**Built-in Scorers:**
- `relevance_to_query/score` — response relevance
- `correctness/score` — factual accuracy
- `safety/score` — safety compliance
- `guidelines/score` — guideline adherence

**Custom Scorers:**
- `response_length_check` — length validation
- `contains_keywords` — keyword presence
- `no_hallucination_markers` — confidence check

**DeepEval Scorers:**
- `conversation_completeness/score` — goal achievement
- `knowledge_retention/score` — context memory
- `topic_adherence/score` — topic focus

### Tips

1. Compare multiple evaluation runs
2. Filter by scorer to find failures
3. Read judge reasoning for insights
4. Track metrics over time for regression detection

---
## Summary

In this notebook, you learned how to evaluate agents and LLM applications using MLflow's evaluation framework.

### What You Learned

**Built-in Scorers:**
- `RelevanceToQuery` - Check if responses are relevant
- `Correctness` - Verify factual accuracy
- `Guidelines` - Enforce custom guidelines
- `Safety` - Ensure safe outputs

**Deterministic Custom Scorers (`@scorer`):**
- Use `@scorer` decorator for code-based evaluation logic
- Access `outputs` and `expectations` in scorers
- Return boolean or numeric scores
- Best for: format validation, keyword checks, structural rules

**LLM-as-a-Judge (`make_judge`):**
- Use `make_judge` for nuanced, LLM-powered evaluation
- Judge receives the full trace via `{{ trace }}` template variable
- Returns a `Feedback` object with `.value` and `.rationale`
- Best for: quality assessment, reasoning evaluation, subjective criteria

**DeepEval Integration:**
- `ConversationCompleteness` - Goal achievement
- `KnowledgeRetention` - Context memory
- `TopicAdherence` - Topic focus

**Session-Level Evaluation:**
- Tag traces with session IDs
- Search traces by session
- Evaluate multi-turn conversations

### Key Patterns

```python
# Built-in scorer
from mlflow.genai.scorers import RelevanceToQuery
scorer = RelevanceToQuery(model="openai:/gpt-4o-mini")

# Deterministic custom scorer
from mlflow.genai import scorer
@scorer
def my_scorer(outputs) -> bool:
    return condition

# LLM-as-a-Judge with make_judge
from mlflow.genai.judges import make_judge
judge = make_judge(
    name="quality_judge",
    instructions="Evaluate the response in {{ trace }}...",
    model="openai:/gpt-4o"
)
feedback = judge(trace=mlflow.get_trace(trace_id))
# feedback.value -> rating, feedback.rationale -> explanation

# Run evaluation
results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[scorer1, scorer2]
)
```

### Next Steps

Continue to **Tutorial 1.8: Prompt Optimization with GEPA** to learn how to automatically optimize prompts using MLflow's GEPA integration.